In [112]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

titanic_df = pd.read_csv(r"C:\Users\yhseo\Downloads\titanic_train.csv")
titanic_df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [114]:
print('## 학습데이터 정보##')
print(titanic_df.info())

## 학습데이터 정보##
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [116]:
titanic_df['Age'].fillna(titanic_df['Age'].mean())
titanic_df['Cabin'].fillna('N')
titanic_df['Embarked'].fillna('N')
print('데이터셋 null값 개수', titanic_df.isnull().sum().sum())

데이터셋 null값 개수 866


In [118]:
print('sex 값 분포', titanic_df['Sex'].value_counts())
print('Cabin 값 분포', titanic_df['Cabin'].value_counts())
print('embarked 값 분포', titanic_df['Embarked'].value_counts())

sex 값 분포 Sex
male      577
female    314
Name: count, dtype: int64
Cabin 값 분포 Cabin
B96 B98        4
G6             4
C23 C25 C27    4
C22 C26        3
F33            3
              ..
E34            1
C7             1
C54            1
E36            1
C148           1
Name: count, Length: 147, dtype: int64
embarked 값 분포 Embarked
S    644
C    168
Q     77
Name: count, dtype: int64


In [120]:
# cabin의 선실번호중 첫번째 알파벳만 추출
titanic_df['Cabin'] = titanic_df['Cabin'].str[:1]
print(titanic_df['Cabin'].head(3))

0    NaN
1      C
2    NaN
Name: Cabin, dtype: object


In [122]:
# 성별이 생존에 어떤 영향을 미치는지
titanic_df.groupby(['Sex','Survived'])['Survived'].count()

Sex     Survived
female  0            81
        1           233
male    0           468
        1           109
Name: Survived, dtype: int64

In [124]:
sns.barplot(x='Sex', y='Survived', data=titanic_df)

<Axes: xlabel='Age_cat', ylabel='Survived'>

In [126]:
sns.barplot(x='Pclass',y='Survived',hue='Sex',data=titanic_df)

<Axes: xlabel='Age_cat', ylabel='Survived'>

In [128]:
# 입력 age에 따라 구분값을 반환하는 함수 설정. DataFrame의 apply lambda 식에 사용
def get_category(age):
    cat=''
    if age <=1: cat= 'Unknown'
    elif age<=5: cat= 'Baby'
    elif age<=12: cat= 'child'
    elif age<=18: cat= 'Teenager'
    elif age<=25: cat= 'student'
    elif age<=35: cat= 'youngadult'
    elif age<=60: cat= 'adult'
    else : cat = 'elderly'

    return cat

# 막대그래프의 크기 figure을 더 크게 설정
plt.figure(figsize=(10,6))

# x축값을 순차적으로 표시하기 위한 설정
group_names = ['Unknown','Baby','child','Teenager','student','youngadult','adult','elderly']

# lambda 식에 위에서 생성한 get_category() 함수를 반환값으로 지정
# get_category(x)는 입력값으로 'Age'칼럼 값을 받아서 해당하는 cat 반환
titanic_df['Age_cat'] = titanic_df['Age'].apply(lambda x: get_category(x))
sns.barplot(x='Age_cat', y='Survived', hue='Sex', data=titanic_df, order=group_names)
titanic_df.drop('Age_cat', axis=1, inplace=True)

In [129]:
from sklearn.preprocessing import LabelEncoder
# labelencoder을 통해 타이타닉 데이터 안에 있는 '글자(문자열)' 데이터를 인공지능이 계산할 수 있도록 '숫자'로 번역해 주는 자동 번역기를 만든 것
def encode_features(dataDF):
    features=['Cabin','Sex','Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(dataDF[feature])
        dataDF[feature] = le.transform(dataDF[feature])
    return dataDF

titanic_df = encode_features(titanic_df)
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,8,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,2,0
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,8,2
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,2,2
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,8,2


In [132]:
# Null 처리함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean())
    df['Cabin'].fillna('N')
    df['Embarked'].fillna('N')
    df['Fare'].fillna(0)
    return df

# 머신러닝 알고리즘에 불필요한 피처 제거
def drop_features(df):
    df.drop(['PassengerId','Name','Ticket'], axis=1,inplace=True)
    return df

# 레이블 인코딩 수행
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin','Sex','Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [134]:
# 원본 데이터를 재로딩하고, 피처데이터세트와 레이블 데이터 세트 추출
titanic_df = pd.read_csv(r"C:\Users\yhseo\Downloads\titanic_train.csv")
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)

X_titanic_df = transform_features(X_titanic_df)

In [136]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state=11)

In [142]:
### 에러없애기 위한 제미나이 코드
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 🌟 1. 에러 안 나는 완벽한 전처리 함수 (inplace=True 모두 제거됨)
def transform_features(df):
    # 결측치 채우기 (다시 변수에 담아주는 최신 방식!)
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Cabin'] = df['Cabin'].fillna('N')
    df['Embarked'] = df['Embarked'].fillna('N')
    df['Fare'] = df['Fare'].fillna(0)
    
    # 불필요한 기둥 버리기
    df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
    
    # 글자를 숫자로 바꾸기
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        df[feature] = le.fit_transform(df[feature]) # fit과 transform을 한 번에!
        
    return df

# 🌟 2. 데이터 불러오기 및 쪼개기 준비
titanic_df = pd.read_csv(r"C:\Users\yhseo\Downloads\titanic_train.csv")
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)

# 🌟 3. 전처리 마법 실행 (여기서 NaN이 완벽히 사라집니다)
X_titanic_df = transform_features(X_titanic_df)

# 확인용 출력: 빈칸이 남아있는지 검사합니다. (모두 0이 나와야 정상!)
print("현재 데이터의 빈칸(NaN) 개수:\n", X_titanic_df.isnull().sum())
print("-" * 30)

# 🌟 4. 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, test_size=0.2, random_state=11)

# 🌟 5. 모델 3대장 출격 및 평가
dt_clf = DecisionTreeClassifier(random_state=11)
rf_clf = RandomForestClassifier(random_state=11)
lr_clf = LogisticRegression(solver='liblinear')

# Decision Tree
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)
print('DecisionTreeClassifier 정확도:', round(accuracy_score(y_test, dt_pred), 4))

# Random Forest
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
print('RandomForestClassifier 정확도:', round(accuracy_score(y_test, rf_pred), 4))

# Logistic Regression
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
print('LogisticRegression 정확도:', round(accuracy_score(y_test, lr_pred), 4))

현재 데이터의 빈칸(NaN) 개수:
 Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Cabin       0
Embarked    0
dtype: int64
------------------------------
DecisionTreeClassifier 정확도: 0.7877
RandomForestClassifier 정확도: 0.8547
LogisticRegression 정확도: 0.8659


In [144]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 결정트리, randomforest, 로지스틱회귀를 위한 사이킷런 classifier 클래스 생성
dt_clf = DecisionTreeClassifier(random_state=11)
rf_clf = RandomForestClassifier(random_state=11)
lr_clf = LogisticRegression(solver='liblinear')

# decisiontreeclassifier 학습/예측/평가
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)
print('decisiontreeclassifier 정확도:', accuracy_score(y_test,dt_pred))

# randomforestclassifier 학습/예측/평가
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
print('randomforestclassifier 정확도:', accuracy_score(y_test,rf_pred))

# logisticregression 학습/예측/평가
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
print('logisticregression 정확도:', accuracy_score(y_test,lr_pred))

decisiontreeclassifier 정확도: 0.7877094972067039
randomforestclassifier 정확도: 0.8547486033519553
logisticregression 정확도: 0.8659217877094972


In [148]:
from sklearn.model_selection import KFold # kfold 클래스를 이용한 교차검증

def exec_kfold(clf, folds=5):
    # 폴드 세트를 5개인 kfold 객체를 생성, 폴드 수만큼 예측결과 저장을 위한 리스트 객체 생성
    kfold = KFold(n_splits=folds)
    scores=[]
    # kfold 교차검증 수행
    for iter_count, (train_index, test_index) in enumerate(kfold.split(X_titanic_df)):
        # X_titanic_df 데이터에서 교차검증별로 학습과 검증 데이터를 가리키는 index 생성
        X_train, X_test = X_titanic_df.values[train_index], X_titanic_df.values[test_index]
        y_train, y_test = y_titanic_df.values[train_index], y_titanic_df.values[test_index]
        # classifier 학습/예측/정확도 계싼
        clf.fit(X_train, y_train)
        predictions = clf.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        scores.append(accuracy)
        print(f'교차검증 {iter_count} 정확도: {accuracy:.4f}')
    # 5개 fold에서의 평균 정확도 계산
    mean_score = np.mean(scores)
    print('평균정확도: ', mean_score)

# exec_kfold 호출
exec_kfold(dt_clf, folds=5)

교차검증 0 정확도: 0.7542
교차검증 1 정확도: 0.7809
교차검증 2 정확도: 0.7865
교차검증 3 정확도: 0.7697
교차검증 4 정확도: 0.8202
평균정확도:  0.782298662984119


In [152]:
from sklearn.model_selection import cross_val_score #cross_val_score는 stratifiedkfold를 이용해 폴드세트를 분할하므로 kfold와 평균정확도가 다름

scores = cross_val_score(dt_clf, X_titanic_df, y_titanic_df, cv=5)

for iter_count, accuracy in enumerate(scores):
    print(f'교차검증 {iter_count} 정확도: {accuracy:.4f}')

print('평균정확도: ', np.mean(scores))

교차검증 0 정확도: 0.7430
교차검증 1 정확도: 0.7753
교차검증 2 정확도: 0.7921
교차검증 3 정확도: 0.7865
교차검증 4 정확도: 0.8427
평균정확도:  0.7879291946519366


In [156]:
from sklearn.model_selection import GridSearchCV # cv는 crossvaildation
# cv는 5개의 폴드를 지정하고 최적 하이퍼파라미터와 그 때의 예측을 출력

parameters = {'max_depth':[2,3,5,10],'min_samples_split':[2,3,5],'min_samples_leaf':[1,5,8]}

grid_dclf = GridSearchCV(dt_clf, param_grid=parameters, scoring='accuracy', cv=5)
grid_dclf.fit(X_train,y_train)

print('gridsearchcv 최적 파라미터:', grid_dclf.best_params_)
print('gridsearchcv 최고 정확도:', grid_dclf.best_score_)
best_dclf = grid_dclf.best_estimator_

# GridSearchCV의 최적 하이퍼파라미터로 학습된 estimator로 예측 및 평가 수행
dpredictions = best_dclf.predict(X_test)
accuracy = accuracy_score(y_test, dpredictions)
print('테스트세트에서의 decisiontreeclassifier 정확도:', accuracy)

gridsearchcv 최적 파라미터: {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 2}
gridsearchcv 최고 정확도: 0.7991825076332119
테스트세트에서의 decisiontreeclassifier 정확도: 0.8715083798882681
